# AutoTranscript - Google Colab

Pipeline pour transcrire un fichier audio francais avec **WhisperX 3.8.6 + pyannote community-1**, GPU T4 et nombre de locuteurs configurable.

Cette version utilise **Google Drive** pour conserver l'audio et les resultats meme si le runtime Colab est reinitialise. Les reglages par defaut privilegient la stabilite pour des entretiens longs.

Avant de commencer :
- activer un **GPU T4** ;
- accepter l'acces Hugging Face a `pyannote/speaker-diarization-community-1` ;
- creer dans **Secrets** un secret `HF_TOKEN` autorise pour ce notebook.


In [ ]:
# 1 - Installation des versions figees
import subprocess
import sys

requirements = '''whisperx==3.8.6
numpy==2.2.6
pyannote-audio==4.0.7
huggingface-hub==0.36.2
'''

with open('/content/requirements-colab.txt', 'w', encoding='utf-8') as f:
    f.write(requirements)

subprocess.run(['apt-get', '-qq', 'update'], check=False)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=False)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/requirements-colab.txt'])

print('Installation terminee.')
print('Si Colab propose de redemarrer l environnement, accepter puis reprendre a la cellule 2.')
print('Sinon, passer directement a la cellule 2.')


Si Colab demande un redemarrage apres l'installation, le faire puis reprendre a la cellule 2. Ne pas relancer la cellule 1 tant que le runtime n'a pas ete reinitialise.


In [ ]:
# 2 - Verification GPU, versions et token Hugging Face
import os
import sys
import importlib.metadata as im

import torch
from google.colab import userdata

assert torch.cuda.is_available(), 'GPU non detecte : activer un runtime GPU T4.'

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN and HF_TOKEN.startswith('hf_'), 'HF_TOKEN absent dans Colab > Secrets.'
os.environ['HF_TOKEN'] = HF_TOKEN

print('Python', sys.version.split()[0])
print('torch', torch.__version__)
print('GPU', torch.cuda.get_device_name(0))
print('whisperx', im.version('whisperx'))
print('pyannote-audio', im.version('pyannote-audio'))
print('numpy', im.version('numpy'))
print('HF_TOKEN : OK')


In [ ]:
# 3 - Monter Google Drive et preparer les dossiers persistants
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/AutoTranscript')
AUDIO_DIR = BASE / 'audio'
RESULTS_DIR = BASE / 'results'

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Dossier audio :', AUDIO_DIR)
print('Dossier resultats :', RESULTS_DIR)
print('Deposer le fichier audio dans Mon Drive/AutoTranscript/audio.')


## Choisir le fichier audio

Si `Mon Drive/AutoTranscript/audio` ne contient qu'un seul fichier audio, la cellule suivante le choisit automatiquement.

S'il y en a plusieurs, remplacer `AUDIO_NAME = None` par le nom exact, par exemple `AUDIO_NAME = "entretien_01.m4a"`.


In [ ]:
# 4 - Selection du fichier audio depuis Google Drive
from pathlib import Path

AUDIO_NAME = None  # ex. 'entretien_01.m4a'

extensions = {'.mp3', '.wav', '.m4a', '.aac', '.flac', '.ogg', '.wma'}
candidates = sorted(
    p for p in AUDIO_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in extensions
)

if AUDIO_NAME:
    AUDIO = AUDIO_DIR / AUDIO_NAME
    assert AUDIO.exists(), f'Fichier introuvable : {AUDIO}'
else:
    assert candidates, f'Aucun fichier audio dans {AUDIO_DIR}'
    assert len(candidates) == 1, (
        'Plusieurs fichiers audio trouves. Mets leur nom exact dans AUDIO_NAME :\n'
        + '\n'.join(' - ' + p.name for p in candidates)
    )
    AUDIO = candidates[0]

print('Audio :', AUDIO.name)
print('Taille : %.1f Mo' % (AUDIO.stat().st_size / 1024**2))


In [ ]:
# 5 - Reglages prudents
MODEL = 'small'      # passer a 'medium' seulement si le test est stable
LANGUAGE = 'fr'
NUM_SPEAKERS = 3
BATCH_SIZE = 4

print('Modele :', MODEL)
print('Langue :', LANGUAGE)
print('Locuteurs :', NUM_SPEAKERS)
print('Batch size :', BATCH_SIZE)


## Test facultatif de 3 minutes

Ce test permet de verifier le token, le GPU et la diarisation avant de lancer tout l'entretien. Si la chaine a deja ete validee, on peut passer directement a la transcription complete.


In [ ]:
# 6 - Test facultatif de 3 minutes
import subprocess
import shutil
from pathlib import Path

TEST_START = 0
TEST_DURATION = 180
TEST_AUDIO = Path('/content/test_3min.wav')
TEST_OUT = Path('/content/test_outputs')

shutil.rmtree(TEST_OUT, ignore_errors=True)
TEST_OUT.mkdir(parents=True, exist_ok=True)

subprocess.check_call([
    'ffmpeg', '-y',
    '-ss', str(TEST_START),
    '-i', str(AUDIO),
    '-t', str(TEST_DURATION),
    '-ar', '16000',
    '-ac', '1',
    str(TEST_AUDIO)
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

cmd = [
    'whisperx', str(TEST_AUDIO),
    '--model', MODEL,
    '--language', LANGUAGE,
    '--diarize',
    '--hf_token', HF_TOKEN,
    '--min_speakers', str(NUM_SPEAKERS),
    '--max_speakers', str(NUM_SPEAKERS),
    '--diarize_model', 'pyannote/speaker-diarization-community-1',
    '--device', 'cuda',
    '--batch_size', str(BATCH_SIZE),
    '--output_dir', str(TEST_OUT),
    '--output_format', 'all'
]

subprocess.check_call(cmd)
print((TEST_OUT / 'test_3min.txt').read_text(encoding='utf-8')[:8000])


## Transcription complete

Les resultats sont ecrits directement dans `Mon Drive/AutoTranscript/results/<nom_du_fichier>/`. Pour un entretien long, garder d'abord `MODEL = 'small'` et `BATCH_SIZE = 4`.

Si le runtime est detruit pendant le traitement, relancer les cellules 2 a 5 puis relancer cette cellule. Les fichiers deja termines restent dans Google Drive.


In [ ]:
# 7 - Transcription complete vers Google Drive
import subprocess
import time
from pathlib import Path

FULL_OUT = RESULTS_DIR / AUDIO.stem
FULL_OUT.mkdir(parents=True, exist_ok=True)

print('Debut de la transcription complete.')
print('Resultats ecrits dans :', FULL_OUT)
start_time = time.time()

cmd = [
    'whisperx', str(AUDIO),
    '--model', MODEL,
    '--language', LANGUAGE,
    '--diarize',
    '--hf_token', HF_TOKEN,
    '--min_speakers', str(NUM_SPEAKERS),
    '--max_speakers', str(NUM_SPEAKERS),
    '--diarize_model', 'pyannote/speaker-diarization-community-1',
    '--device', 'cuda',
    '--batch_size', str(BATCH_SIZE),
    '--output_dir', str(FULL_OUT),
    '--output_format', 'all'
]

subprocess.check_call(cmd)

elapsed = time.time() - start_time
print('Termine en %.1f minutes.' % (elapsed / 60))
print('Resultats :')
for p in sorted(FULL_OUT.iterdir()):
    print(' -', p.name)


In [ ]:
# 8 - Produire un TXT horodate adapte a l'analyse
import json
from pathlib import Path

json_files = sorted(FULL_OUT.glob('*.json'))
assert json_files, 'Aucun JSON WhisperX trouve.'

json_file = json_files[0]
data = json.loads(json_file.read_text(encoding='utf-8'))
mapping = {}

def hhmmss(x):
    x = int(x)
    h, r = divmod(x, 3600)
    m, s = divmod(r, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'

lines = []
for seg in data.get('segments', []):
    raw = seg.get('speaker', 'INCONNU')
    if raw not in mapping:
        mapping[raw] = f'Locuteur {len(mapping)+1}'
    lines.append(
        f"{mapping[raw]} [{hhmmss(seg.get('start', 0))} - {hhmmss(seg.get('end', 0))}] : "
        f"{seg.get('text', '').strip()}"
    )

research_txt = FULL_OUT / (AUDIO.stem + '_recherche.txt')
research_txt.write_text('\n\n'.join(lines), encoding='utf-8')

print('TXT recherche :', research_txt)
print('Apercu :')
print(research_txt.read_text(encoding='utf-8')[:8000])


In [ ]:
# 9 - Optionnel : creer une archive ZIP dans Google Drive
import shutil

archive_base = str(FULL_OUT.parent / (AUDIO.stem + '_transcription'))
archive = shutil.make_archive(archive_base, 'zip', FULL_OUT)

print('Archive creee :', archive)


## En cas de deconnexion Colab

- Simple reconnexion au meme runtime : ne rien reinstaller ; reprendre la cellule interrompue.
- Runtime reinitialise : relancer la cellule 1 si necessaire, puis les cellules 2 a 5 et enfin la transcription complete.
- L'audio et les resultats deja ecrits restent dans Google Drive.
